# Forecasting Training Notebook

This notebook trains a Prophet forecasting model on the ingested `backend/data/trends.csv` dataset and saves a serialized model to `backend/forecast/prophet_model.joblib`.

Cells are numbered below. Run them sequentially to reproduce training.

In [ ]:
# 1. Imports and setup
import os
import pandas as pd
from prophet import Prophet
import joblib
print('pandas', pd.__version__)

In [ ]:
# 2. Load the sample CSV (produced by ingestion)
csv_path = os.path.normpath(os.path.join('..', 'data', 'trends.csv'))
print('Looking for', csv_path)
if not os.path.exists(csv_path):
    print('CSV not found, creating synthetic series for demo')
    today = pd.Timestamp.today().normalize()
    dates = [today - pd.Timedelta(days=7 * i) for i in range(8)][::-1]
    ts = pd.DataFrame({'ds': dates, 'y': [100 + i * 5 for i in range(len(dates))]})
else:
    df = pd.read_csv(csv_path)
    if 'date' in df.columns and 'metric' in df.columns:
        df['ds'] = pd.to_datetime(df['date']).dt.normalize()
        ts = df.groupby('ds').metric.median().reset_index().rename(columns={'metric': 'y'})
    else:
        # fallback synthetic series
        today = pd.Timestamp.today().normalize()
        dates = [today - pd.Timedelta(days=7 * i) for i in range(8)][::-1]
        ts = pd.DataFrame({'ds': dates, 'y': [100 + i * 5 for i in range(len(dates))]})

print(ts.head())

In [ ]:
# 3. Train Prophet and save the model
m = Prophet(daily_seasonality=False)
m.fit(ts)
out_path = os.path.normpath(os.path.join(os.path.dirname(__file__), 'prophet_model.joblib'))
joblib.dump(m, out_path)
print('Saved model to', out_path)

In [ ]:
# 4. Quick forecast check
future = m.make_future_dataframe(periods=7)
fc = m.predict(future)
print(fc[['ds','yhat']].tail(7))